# Construisez un projet de Machine Learning:

## Partie 4 | `Suivi des expériences`

Dans ce notebook, nous allons démontrer le **suivi des expériences** à l’aide des capacités de suivi d’expériences de Snowflake ML avec le jeu de données de **classification des espèces d’ours**. Nous allons entraîner plusieurs modèles et comparer leurs performances en utilisant les fonctionnalités intégrées de suivi d’expériences de Snowflake.

### Ce que nous allons couvrir :

1. **Chargement et préparation des données** - Charger et traiter le jeu de données des ours à l’aide de Snowpark (`snowflake-snowpark-python`)
2. **Configuration de l’expérience** - Initialiser le suivi des expériences Snowflake ML (`ExperimentTracking()` depuis `snowflake-ml-python`)
3. **Entraînement des modèles** - Entraîner plusieurs modèles dans le cadre de l’ajustement des hyperparamètres avec `scikit-learn` en utilisant l’algorithme Random Forest. Les performances sont suivies.
4. **Comparaison des performances** - Comparer les modèles à l’aide des métriques suivies
5. **Sélection du modèle** - Sélectionner le modèle le plus performant et l’enregistrer dans le registre de modèles Snowflake (`log_model()` depuis `snowflake-ml-python`)

## Installer les bibliothèques prérequises

Les Snowflake Notebooks incluent par défaut les bibliothèques Python courantes. Pour en ajouter d’autres, utilisez le menu déroulant **Packages** en haut à droite. 

Ajoutons les packages suivants :
- `modin` - Effectuer des opérations sur les données (lecture/écriture) et du data wrangling comme avec pandas via l’[API Snowpark pandas](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/latest/modin/index)
- `scikit-learn` - Effectuer des divisions de données et construire des modèles de machine learning
- `snowflake-ml-python` - un ensemble de fonctionnalités ML fournies par Snowflake. Ici, nous utiliserons la fonctionnalité de journalisation des métriques de modèle.


In [ ]:
from snowflake.snowpark.context import get_active_session
import streamlit as st

# Get active Snowflake session 
session = get_active_session()
st.write("✅ Connected using active Snowflake session!")

In [ ]:
import warnings

# Filter out ResourceWarning
warnings.filterwarnings('ignore', category=ResourceWarning)

# Filter out DeprecationWarning
warnings.filterwarnings('ignore', category=DeprecationWarning)

# Filter out UserWarning
warnings.filterwarnings('ignore', category=UserWarning)


### Charger les données

Enfin, nous allons charger le jeu de données des ours.

In [ ]:
# Load bear dataset from Snowflake
import pandas as pd

bear_df = session.table('BEARS_DATA.STAGES.BEAR').to_pandas()

st.write("📊 Bear Dataset Loaded:")
st.write(f"Shape: {bear_df.shape}")

bear_df

## 2. Préparation des données

### Préparer les variables explicatives et la variable cible

In [ ]:
# Prepare features and target
from sklearn.model_selection import train_test_split

X = bear_df.drop(columns=['species', 'id'])
y = bear_df['species']

### Données manquantes

Il est toujours recommandé de vérifier la présence de données manquantes.

In [ ]:
# Check for missing data
missing_features = X.isnull().sum().sum()
missing_target = y.isnull().sum()

st.subheader("🔍 Data Quality Check:")
st.write(f"Missing feature values: `{missing_features}`")
st.write(f"Missing target values: `{missing_target}`")

### Découpage des données

Ici, nous allons diviser les données en ensembles d’entraînement et de test en utilisant une répartition 80/20.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

st.write("✅ Data preparation completed!")

st.subheader("📊 Data Split Summary:")
st.write(f"Training set: `{X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.1f}%)`")
st.write(f"Testing set: `{X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.1f}%)`")
st.write(f"Number of features: {X_train.shape[1]}")

st.subheader("🎯 Class Distribution:")
st.write(f"Training set: `{y_train.value_counts().sort_index().to_dict()}`")
st.write(f"Testing set: `{y_test.value_counts().sort_index().to_dict()}`")

### Mise à l’échelle des variables

Pour préparer nos données à l’entraînement du modèle, nous allons appliquer les prétraitements suivants :
- `StandardScaler` pour les variables numériques. Cela transforme les variables afin qu’elles aient une moyenne = 0 et un écart-type = 1
- `OneHotEncoder` pour les variables catégorielles (`fur_color`, `facial_profile`, `paw_pad_texture`). Cela convertit les variables catégorielles en colonnes binaires.


In [ ]:
# Feature scaling and preprocessing
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Identify numerical and categorical columns
numerical_features = X_train.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X_train.select_dtypes(include=['object']).columns

# Scale numerical features
scaler = StandardScaler()
X_train_scaled_num = scaler.fit_transform(X_train[numerical_features])
X_test_scaled_num = scaler.transform(X_test[numerical_features])

# Handle categorical features
onehot = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_train_scaled_cat = onehot.fit_transform(X_train[categorical_features])
X_test_scaled_cat = onehot.transform(X_test[categorical_features])

# Get feature names after one-hot encoding and replace spaces with underscores
cat_feature_names = []
for feature, categories in zip(categorical_features, onehot.categories_):
    for category in categories:
        cat_feature_names.append(f"{feature}_{category}".replace(' ', '_').lower())

# Combine numerical and categorical features
X_train_scaled = np.hstack([X_train_scaled_num, X_train_scaled_cat])
X_test_scaled = np.hstack([X_test_scaled_num, X_test_scaled_cat])

# Convert to DataFrame with proper column names
all_feature_names = list(numerical_features) + cat_feature_names
X_train_scaled = pd.DataFrame(X_train_scaled, columns=all_feature_names, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=all_feature_names, index=X_test.index)

st.write("✅ Feature scaling completed!")

st.subheader("🔧 Feature Processing:")
st.write(f"Numerical features: `{numerical_features.tolist()}`")
st.write(f"Categorical features: `{categorical_features.tolist()}`")

st.subheader("📊 Scaled Data Dimensions:")
st.write(f"Training features: `{X_train_scaled.shape}`")
st.write(f"Testing features: `{X_test_scaled.shape}`")

# Display first few encoded feature names
st.write("\n🏷️ First few encoded feature names:")
st.write(all_feature_names[:10])


### Encoder la variable cible

Nous allons également encoder la variable cible (espèce d’ours) en valeurs numériques à l’aide du `LabelEncoder` de scikit-learn.
- Les modèles de machine learning nécessitent des entrées numériques
- Chaque espèce d’ours unique se verra attribuer une valeur entière unique
- Cet encodage conserve la nature catégorielle des espèces tout en les rendant adaptées à l’entraînement du modèle

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Encode target variable
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)


In [ ]:
y_train_encoded

In [ ]:
y_test_encoded

### Écrire les données de test dans une table

Écrivons maintenant les données de l’ensemble de test (stockées dans la variable `X_test_scaled`) dans une table Snowflake `BEAR_TEST_DATA`

In [ ]:
# Create a copy of X_test_scaled
test_df = X_test_scaled.copy()

# Add the encoded target variable
test_df['ACTUAL_SPECIES'] = y_test_encoded

# Convert to Snowpark DataFrame and write to table
snowpark_df = session.create_dataframe(test_df)
snowpark_df.write.mode("overwrite").save_as_table("BEARS_DATA.STAGES.BEAR_TEST_DATA")

st.write("✅ Data successfully saved to BEAR_TEST_DATA table!")
st.write(f"Number of rows: {len(test_df)}")
st.write(f"Number of columns: {len(test_df.columns)}")


## 3. Configuration du suivi des expériences

Configurons maintenant notre suivi des expériences en utilisant les capacités de suivi des expériences de Snowflake ML. Cela nous permettra d’enregistrer et de comparer systématiquement différents modèles ainsi que leurs hyperparamètres.

Nous allons commencer par créer le tracker d’expériences avec `ExperimentTracking` du package Snowflake ML.

In [ ]:
from snowflake.ml.experiment.experiment_tracking import ExperimentTracking

# Create ExperimentTracking
exp = ExperimentTracking(session=session)

# Set Experiment Name
experiment_name = "Bear_Classification_Experiment"
exp.set_experiment(experiment_name)

st.write(f"✅ Experiment Tracking Initialized: `{experiment_name}`")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, matthews_corrcoef
from datetime import datetime

# Define Hyperparameters ---
params = {
    "n_estimators": 100,
    "max_depth": 3,
    "min_samples_leaf": 5,
    'max_features': 'sqrt',
    "random_state": 42
}

# Create unique run name with timestamp
run_name = f"bear_baseline_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

# Train, Evaluate and Log Model
with exp.start_run(run_name):
    # Log hyperparameters
    exp.log_params(params)

    # Train model
    model = RandomForestClassifier(**params)
    model.fit(X_train_scaled, y_train_encoded)

    # Predict
    y_pred = model.predict(X_test_scaled)

    # Calculate metrics
    acc = accuracy_score(y_test_encoded, y_pred)
    precision = precision_score(y_test_encoded, y_pred, average='macro')
    recall = recall_score(y_test_encoded, y_pred, average='macro')
    mcc = matthews_corrcoef(y_test_encoded, y_pred)

    # Log metrics
    exp.log_metric("accuracy", acc)
    exp.log_metric("precision", precision)
    exp.log_metric("recall", recall)
    exp.log_metric("mcc", mcc)

    # Display results
    st.write("📊 Model Performance:")
    st.write(f"- Accuracy: `{acc:.4f}`")
    st.write(f"- Precision: `{precision:.4f}`")
    st.write(f"- Recall: `{recall:.4f}`")
    st.write(f"- MCC: `{mcc:.4f}`")


## 4. Ajustement des hyperparamètres

Analysons maintenant les résultats de nos expériences d’ajustement des hyperparamètres de Random Forest. Nous allons récupérer les métriques enregistrées et examiner en détail le processus d’ajustement.


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, matthews_corrcoef
from datetime import datetime
import itertools
import pandas as pd

# Define hyperparameter grid
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5],
    "min_samples_leaf": [1, 2, 5],
    "max_features": ['sqrt', 'log2']
}

# Initialize list to store results
results = []

# Generate all combinations of parameters
param_combinations = [dict(zip(param_grid.keys(), v)) for v in itertools.product(*param_grid.values())]

# Train models with different parameters
for params in param_combinations:
    # Add random state to params
    params['random_state'] = 42
    
    # Create unique run name with timestamp and params summary
    run_name = f"RF_tune_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

    with exp.start_run(run_name):
        # Log hyperparameters
        exp.log_params(params)

        # Train model
        model = RandomForestClassifier(**params)
        model.fit(X_train_scaled, y_train_encoded)

        # Predict
        y_pred = model.predict(X_test_scaled)

        # Calculate metrics
        acc = accuracy_score(y_test_encoded, y_pred)
        precision = precision_score(y_test_encoded, y_pred, average='macro')
        recall = recall_score(y_test_encoded, y_pred, average='macro')
        mcc = matthews_corrcoef(y_test_encoded, y_pred)

        # Log metrics
        exp.log_metric("accuracy", acc)
        exp.log_metric("precision", precision)
        exp.log_metric("recall", recall)
        exp.log_metric("mcc", mcc)

        # Store results
        results.append({
            'run_name': run_name,
            'n_estimators': params['n_estimators'],
            'max_depth': params['max_depth'],
            'min_samples_leaf': params['min_samples_leaf'],
            'max_features': params['max_features'],
            'accuracy': acc,
            'precision': precision,
            'recall': recall,
            'mcc': mcc
        })

        st.write(f"Parameters: {params}")

Maintenant que nous avons exécuté l’ajustement des hyperparamètres, analysons les résultats afin de trouver le modèle le plus performant.

In [ ]:
# Create DataFrame from results
results_df = pd.DataFrame(results)

# Display summary statistics
st.write("📊 Model Performance Summary:")
st.dataframe(results_df.style.highlight_max(subset=['accuracy', 'precision', 'recall', 'mcc'], color="green"))

# Display best performing configuration
best_model = results_df.loc[results_df['accuracy'].idxmax()]

st.subheader("🏆 Best Model Configuration:")
st.write("Learning Algorithm: `Random Forest`")
st.write(f"Accuracy: `{best_model['accuracy']:.4f}`")
st.write(f"Precision: `{best_model['precision']:.4f}`")
st.write(f"Recall: `{best_model['recall']:.4f}`")
st.write(f"MCC: `{best_model['mcc']:.4f}`")
st.write(f"Learning Parameters:")
st.write(f"- n_estimators: `{best_model['n_estimators']}`")
st.write(f"- max_depth: `{best_model['max_depth']}`")
st.write(f"- min_samples_leaf: `{best_model['min_samples_leaf']}`")
st.write(f"- max_features: `{best_model['max_features']}`")


## 5. Registre de modèles

Enregistrons maintenant le meilleur modèle dans le registre de modèles de Snowflake pour le déploiement et la gestion.


### Entraîner le modèle final avec les meilleurs paramètres

Maintenant que nous avons identifié la configuration de modèle la plus performante à partir de nos expériences d’ajustement des hyperparamètres, entraînons le modèle final en utilisant ces paramètres optimaux :

- Nombre d’estimateurs : 100
- Profondeur maximale : 5
- Nombre minimum d’échantillons par feuille : 2
- Nombre maximal de variables : sqrt

In [ ]:
# Get best model configuration from previous results
best_params = {
    'n_estimators': int(best_model['n_estimators']),
    'max_depth': int(best_model['max_depth']),
    'min_samples_leaf': int(best_model['min_samples_leaf']),
    'max_features': best_model['max_features'],
    'random_state': 42
}

# Train the best model
final_model = RandomForestClassifier(**best_params)
final_model.fit(X_train_scaled, y_train_encoded)

In [ ]:
best_params

### Enregistrer le meilleur modèle

Nous allons maintenant enregistrer notre modèle Random Forest le plus performant dans le registre de modèles de Snowflake.

In [ ]:
# Create model registry instance
from snowflake.ml.registry import Registry
from datetime import datetime
import warnings

# Temporarily suppress warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)

registry = Registry(session)

# Clean the column names by replacing spaces with underscores
X_train_clean = X_train_scaled.copy()
X_train_clean.columns = X_train_clean.columns.str.replace(' ', '_')

# Register model
model_name = "BEAR_SPECIES_CLASSIFIER"
model_version = f"v_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

model_ref = registry.log_model(
    model=final_model,
    model_name=model_name,
    version_name=model_version,
    sample_input_data=X_train_clean.head(5),
    metrics={
        'accuracy': float(best_model['accuracy']),
        'precision': float(best_model['precision']),
        'recall': float(best_model['recall']),
        'mcc': float(best_model['mcc'])
    },
    options={
        "case_sensitive": True,
        "min_positive_value": 1e-10  # Add small constant to prevent log(0)
    },
    comment="Best performing Random Forest model from hyperparameter tuning"
)

# Reset warnings to default
warnings.resetwarnings()

st.write("✅ Model successfully registered!")
st.write(f"Model Name: `{model_name}`")
st.write(f"Version: `{model_version}`")


### Afficher les modèles dans le registre

In [ ]:
SHOW MODELS

In [ ]:
registry.show_models()

### Afficher les versions disponibles dans un modèle

In [ ]:
SHOW VERSIONS IN MODEL bear_species_classifier;

### Afficher les fonctions disponibles dans un modèle

In [ ]:
SHOW FUNCTIONS IN MODEL bear_species_classifier

## Déployer le modèle en tant que service

In [ ]:
# First drop existing service using SQL through session
session.sql("DROP SERVICE IF EXISTS bear_rf_classifier").collect()

# Deploy to a CPU compute pool on SPCS
model_ref.create_service(
    service_name="bear_rf_classifier",
    service_compute_pool="system_compute_pool_cpu",
    ingress_enabled=True,
    gpu_requests=None
)

st.write("✅ Model service created successfully!")

### Afficher les endpoints du service

Examinons les endpoints exposés par notre modèle déployé.

In [ ]:
SHOW SERVICES;

In [ ]:
SHOW ENDPOINTS IN SERVICE bear_rf_classifier;

### Effectuer une inférence du modèle

Maintenant que notre modèle est déployé en tant que service, nous pouvons l’utiliser pour effectuer des prédictions sur de nouvelles données. 

Voici ce que nous faisons :
- Interroger la table `BEAR_TEST_DATA`
- Faire passer les variables dans notre modèle déployé
- Obtenir des prédictions pour la classification des espèces d’ours à l’aide de la fonction `PREDICT()`

In [ ]:
SELECT
  BEAR_RF_CLASSIFIER ! PREDICT(
    "body_mass_kg",
    "shoulder_hump_height_cm",
    "claw_length_cm",
    "snout_length_cm",
    "forearm_circumference_cm",
    "ear_length_cm",
    "fur_color_black",
    "fur_color_blackish_brown",
    "fur_color_blond",
    "fur_color_brown",
    "fur_color_cinnamon",
    "fur_color_dark_brown",
    "fur_color_grizzled",
    "fur_color_light_brown",
    "fur_color_medium_brown",
    "fur_color_reddish_brown",
    "facial_profile_dished",
    "facial_profile_straight",
    "paw_pad_texture_rough",
    "paw_pad_texture_smooth"
  ) AS predicted_species
FROM
  BEARS_DATA.STAGES.BEAR_TEST_DATA
LIMIT
  5;